# I-S101 — 안정화 Dual-Encoder Gated SegResNet

I-S100의 구조는 유지하되 수치 붕괴를 방지하는 재실험이다. I-S100 결과와 checkpoint는 수정하지 않는다.

변경점: learning rate `1e-3 → 3e-4`, AMP 비활성화, gradient clipping `max_norm=1.0`, loss/gradient NaN 즉시 중단, epoch별 gradient norm 기록. 나머지 데이터 split, sampling, 300 epochs, full-volume validation, Top-3 ±5 선택 및 확률맵 평가는 동일하다.

In [ ]:
from pathlib import Path
import json, os, signal, subprocess, time
import pandas as pd
from IPython.display import display, clear_output

REPO=Path("/home/stu03/workspace/model")
PYTHON=REPO/".venv/bin/python"
CONFIG=REPO/"configs/experiments/ischemia/I-S101.yaml"
RUN=REPO/"runs/I-segresnet/I-S101"
LOG=RUN/"sequence.log"
PID_FILE=RUN/"sequence.pid"
assert PYTHON.is_file() and CONFIG.is_file()

## 1. 실행 명령 확인

In [ ]:
subprocess.run([str(PYTHON),str(REPO/"scripts/ischemia/training/run_i_s_experiment.py"),"--config",str(CONFIG),"--dry-run"],cwd=REPO,check=True)

## 2. 실제 batch sanity test

학습은 시작하지 않고 모델 출력 shape와 finite loss만 확인한다.

In [ ]:
subprocess.run([str(PYTHON),"-m","brainon_ai.models.ischemia.segresnet_hy.train","--config",str(CONFIG),"--data-root","/home/stu03/workspace/datasets/ischemic/SegResNet","--backup-root","/tmp/I-S101_sanity","--sanity-only"],cwd=REPO,env={**os.environ,"PYTHONPATH":str(REPO/"src")},check=True)

## 3. 학습 시작/재개

기존 `checkpoint_latest.pth`가 있으면 자동 재개한다.

In [ ]:
RUN.mkdir(parents=True,exist_ok=True)
if PID_FILE.is_file():
    old=int(PID_FILE.read_text())
    try: os.kill(old,0); raise RuntimeError(f"이미 실행 중: PID={old}")
    except ProcessLookupError: pass
handle=LOG.open("a",buffering=1)
cmd=[str(PYTHON),str(REPO/"scripts/ischemia/training/run_i_s_experiment.py"),"--config",str(CONFIG)]
proc=subprocess.Popen(cmd,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT,start_new_session=True)
PID_FILE.write_text(str(proc.pid)); print(f"started PID={proc.pid} | {LOG}")

## 4. 실시간 진행률

10초마다 갱신한다. 모니터 셀을 중단해도 background 학습은 계속된다. 완료·실패 상태에서는 자동 종료한다.

In [ ]:
def alive():
    if not PID_FILE.is_file(): return False
    try: os.kill(int(PID_FILE.read_text()),0); return True
    except (ProcessLookupError,ValueError): return False
try:
    while True:
        progress=json.loads((RUN/"progress.json").read_text()) if (RUN/"progress.json").is_file() else {"status":"waiting"}
        clear_output(wait=True); print(time.strftime("Updated: %Y-%m-%d %H:%M:%S")); display(pd.DataFrame([progress]))
        numeric=RUN/"logs/numerics_log.csv"
        if numeric.is_file(): display(pd.read_csv(numeric).tail(10))
        if LOG.is_file(): print("\n--- latest log ---\n"+"".join(LOG.read_text(errors="replace").splitlines(True)[-20:]))
        terminal=progress.get("status") in {"completed","failed_nonfinite_loss","failed_nonfinite_gradient"}
        if terminal or not alive(): print("\n실행 종료. 마지막 상태입니다."); break
        time.sleep(10)
except KeyboardInterrupt: print("모니터만 중단했습니다. background 학습은 계속됩니다.")

## 5. 중단 — 필요할 때만 실행

In [ ]:
pid=int(PID_FILE.read_text()); os.killpg(pid,signal.SIGTERM); print(f"stop requested: {pid}")

## 6. 최종 결과

In [ ]:
selection=RUN/"final_selection.json"; sweep=RUN/"final_metrics/threshold_sweep.csv"
if not selection.is_file(): print("최종 선택 결과가 아직 없습니다.")
else:
    print(json.dumps(json.loads(selection.read_text())["selected"],ensure_ascii=False,indent=2))
    if sweep.is_file():
        df=pd.read_csv(sweep); display(df.sort_values(["lesion_f2","macro_dice_positive"],ascending=False).head(10))